# Exploration: where the 0.95 AUC goes

This dataset is routinely reported with an AUC above 0.90. This notebook takes that number
apart on real data and shows what is left once the model is only allowed to know what it
would know at dialling time.

Two leaks, measured one at a time.

In [1]:
import sys
sys.path.insert(0, "src")

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from lead_scoring.config import CATEGORICAL, MACRO_COLUMNS, NUMERIC
from lead_scoring.data import chronological_split, load

full = load(drop_leakage=False)     # keeps `duration`
clean = load(drop_leakage=True)
print(f"{len(full):,} contacts, {full['converted'].mean():.1%} converted overall")

41,188 contacts, 11.3% converted overall


## 1. Leak one: `duration`

`duration` is how long the call lasted. It exists only after the call is over, and a long
call is a call that went well. At scoring time — before anyone has dialled — the field does
not exist.

In [2]:
converted = full.groupby("converted")["duration"].median()
print("median call length")
print(f"  did not subscribe  {converted[0]:6.0f} seconds")
print(f"  subscribed         {converted[1]:6.0f} seconds")
print(f"\nratio: {converted[1] / converted[0]:.1f}x")

median call length
  did not subscribe     164 seconds
  subscribed            449 seconds

ratio: 2.7x


## 2. Leak two: the macroeconomic columns

Five columns describe the economy on the day of the call. Two problems, and the second only
shows up under a time split.

First: every lead called on the same day shares these values. A feature constant across the
items being ranked cannot rank them.

In [3]:
train, _, test = chronological_split(clean)

print(f"{'column':<18}{'train range':>24}{'test range':>24}   overlap")
for column in MACRO_COLUMNS:
    train_range = f"{train[column].min():.2f} .. {train[column].max():.2f}"
    test_range = f"{test[column].min():.2f} .. {test[column].max():.2f}"
    overlaps = test[column].min() <= train[column].max() and test[column].max() >= train[column].min()
    print(f"{column:<18}{train_range:>24}{test_range:>24}   {'yes' if overlaps else 'NONE'}")

column                         train range              test range   overlap
emp.var.rate                 -0.10 .. 1.40          -3.40 .. -1.10   NONE
cons.price.idx              93.20 .. 94.47          92.20 .. 94.77   yes
cons.conf.idx             -42.70 .. -36.10        -50.80 .. -26.90   yes
euribor3m                     4.08 .. 5.04            0.63 .. 1.30   NONE
nr.employed             5191.00 .. 5228.10      4963.60 .. 5099.10   NONE


`euribor3m` has **zero overlap**: the campaign runs through the 2008 crisis, rates collapse,
and the test period lives entirely outside the range the model was trained on. Trees cannot
extrapolate — a split learned at "euribor > 4" meets nothing but euribor < 1.3.

In [4]:
share = test["euribor3m"].isin(set(train["euribor3m"])).mean()
print(f"test euribor3m values also seen in training: {share:.1%}")
print()
print(f"conversion rate, training period: {train['converted'].mean():.1%}")
print(f"conversion rate, test period:     {test['converted'].mean():.1%}")

test euribor3m values also seen in training: 0.0%

conversion rate, training period: 4.8%
conversion rate, test period:     30.8%


## 3. Measuring each leak

In [5]:
MACRO = list(MACRO_COLUMNS)
ALL_COLS = list(CATEGORICAL) + list(NUMERIC)

def prepare(frame, columns):
    out = frame[columns].copy()
    for column in columns:
        if column in CATEGORICAL:
            out[column] = out[column].astype("category")
    return out

def score(frame, columns, split):
    if split == "chronological":
        tr, _, te = chronological_split(frame)
    else:
        tr, te = train_test_split(frame, test_size=0.2, random_state=42, stratify=frame["converted"])
    model = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                           min_child_samples=50, random_state=42, verbose=-1)
    model.fit(prepare(tr, columns), tr["converted"])
    return roc_auc_score(te["converted"], model.predict_proba(prepare(te, columns))[:, 1])

results = []
for split in ("random", "chronological"):
    results.append({"split": split, "features": "duration + macro",
                    "auc": score(full, ALL_COLS + MACRO + ["duration"], split)})
    results.append({"split": split, "features": "macro, no duration",
                    "auc": score(clean, ALL_COLS + MACRO, split)})
    results.append({"split": split, "features": "neither (deployable)",
                    "auc": score(clean, ALL_COLS, split)})

print(pd.DataFrame(results).pivot(index="features", columns="split", values="auc").round(3).to_string())

split                 chronological  random
features                                   
duration + macro              0.728   0.954
macro, no duration            0.597   0.811
neither (deployable)          0.640   0.778


## What the table says

Read the **chronological** column — the one that mimics production, where you train on the
past and score the future.

Dropping the macro columns **raises** chronological AUC while **lowering** random-split AUC.
That asymmetry is the signature of a time proxy: a feature that helps when the model is
allowed to peek at the period, and hurts when it is not.

The deployable number is around **0.64**, not 0.95. What that buys is a 1.54x lift on call
ordering, which is the business case — see the README.